In [ ]:
!pwd

In [ ]:
from unimol_tools import MolTrain, MolPredict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr
import os

In [ ]:
# 更改工作目录到 `af` 目录
os.chdir('/data2/Jn/Unimol')

# 验证当前工作目录
print(os.getcwd())

In [ ]:
weights_path = r"./weights_for_props_prediction/lumo/bs_32_lr_1e-4"
data_path = r"./SDF_SMILES_Prop/TestFinal_test_lumo.csv"
save_path = r"./Pred_props_assess/TestFinal_lumo_pred.csv"
input_csv = r"./tmp.csv"

In [ ]:
data = pd.read_csv(data_path)
data.head(5)

In [ ]:
pd.DataFrame(data.iloc[:,0]).to_csv(input_csv, index=False)

In [ ]:
clf = MolPredict(load_model=weights_path)

In [ ]:
# 基于SMILES的文件输入模式的预测 
test_pred = clf.predict(data=input_csv)

In [ ]:
os.remove(input_csv)

In [ ]:
test_results = pd.DataFrame({
    "smiles": clf.datahub.data["smiles"],
    "pred": test_pred.flatten(),
    "real": data.iloc[:, 1]
    })
test_results.head(10)

In [ ]:
test_results.to_csv(save_path, index=False)

In [ ]:
true_values = test_results["real"]
predicted_values = test_results["pred"]

In [ ]:
# 计算x轴和y轴的范围
x_min = math.floor(min(true_values))
x_max = math.ceil(max(true_values))
y_min = math.floor(min(predicted_values))
y_max = math.ceil(max(predicted_values))

# 统一x轴和y轴范围，选择最广的范围
min_range = min(x_min, y_min)
max_range = max(x_max, y_max)

# 设置绘图风格和自定义参数，替换字体为 DejaVu Sans
plt.style.use('default')
plt.rcParams.update({
    'font.size': 22,                   # 字号
    'font.family': 'DejaVu Sans',      # 替换为系统中可用的字体
    'axes.linewidth': 2,               # 边框粗细
    'xtick.major.size': 8,             # x轴刻度长度
    'ytick.major.size': 8,             # y轴刻度长度
    'xtick.major.width': 2,            # x轴刻度粗细
    'ytick.major.width': 2,            # y轴刻度粗细
    'legend.frameon': False,           # 去掉图例边框
})

# 创建散点图
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
plt.scatter(true_values, predicted_values, color='cornflowerblue', edgecolor='royalblue', s=50, alpha=0.8, zorder=3)  # 设置填充颜色和边框颜色

# 添加对角线 y=x
plt.plot([min_range, max_range], [min_range, max_range], color='black', linewidth=2, linestyle='--', zorder=2)

# 添加误差范围线 y=x±0.16
plt.plot([min_range, max_range], [min_range + 0.1, max_range + 0.1], color='darkorange', linewidth=2, linestyle='--', alpha=0.7, zorder=4)
plt.plot([min_range, max_range], [min_range - 0.1, max_range - 0.1], color='darkorange', linewidth=2, linestyle='--', alpha=0.7, zorder=4)

# 设置统一的轴范围
plt.xlim(min_range, max_range)
plt.ylim(min_range, max_range)

# 设置标签
plt.xlabel('Truth')
plt.ylabel('Prediction')

# 添加图例
plt.legend()

# 显示图表
plt.show()

In [ ]:
# 计算误差
errors = np.abs(true_values - predicted_values)

# 计算误差小于0.1的点的数量
within_error = np.sum(errors < 0.1)

# 计算误差小于0.16的点所占的百分比
percentage_within_error = (within_error / len(true_values)) * 100
print(f'Percentage of points with error < 0.1: {percentage_within_error:.2f}%')

In [ ]:
# 计算平均绝对误差 (MAE)
mae = mean_absolute_error(true_values, predicted_values)
print(f'Mean Absolute Error (MAE): {mae:.4f}')

# 计算 Spearman 相关系数
spearman_corr, spearman_p = spearmanr(true_values, predicted_values)
print(f'Spearman Correlation: {spearman_corr:.4f}, p-value: {spearman_p:.4f}')

# 计算 R² (决定系数)
r2 = r2_score(true_values, predicted_values)
print(f'R² (Coefficient of Determination): {r2:.4f}')